# 02 · Campaign — two-paradigm binder design vs PD-L1

**Standard slot:** *design campaign.* **For Project 06 this is the core:** run **both** paradigms
against the PD-1-face hotspots and assemble their pools (D2):
- **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.

Then score every design with **AF2-Multimer** (`pae_interaction` is the key binder metric).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster).
> Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a small RFdiffusion batch +
> ESMFold triage). The cells below run on the deterministic **mock** backend so the plumbing executes
> anywhere; the real calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log
it). The generation itself needs an A100; this check needs nothing.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # e.g. pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

## 1 · Define the campaign

Same target + hotspots as notebook 01. Set honest campaign sizes; the cells run on `mock` so they
execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the numbers on
a T4** (FreeBindCraft, a small RFdiffusion batch).

In [ ]:
import binder_tools as bt
import pandas as pd

TARGET = "PDL1"
HOTSPOTS = bt.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with your verified PD-1-face residues

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"AF2-Multimer: tool={TOOL_AF2}")
print("hotspots    :", HOTSPOTS)

## 2 · Paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/binder_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

## 3 · Paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the hotspots, then ProteinMPNN designs sequences, then
AF2-Multimer re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate
is low — that is normal). The `mock` backend stands in for the whole chain.

In [ ]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

## 4 · Assemble + persist both pools

Write one tidy CSV per paradigm (plus a combined one). These feed notebook 03 (the shared filter).
We add an EXAMPLE physics column (`rosetta_dG`) here so the binder physics layer has something to act
on in the dry run — on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [ ]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

## D2 checklist
- [ ] BindCraft pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4).
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100).
- [ ] Every design scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on both pools.